# YOLOv8 蒸馏训练（yolov8n 学生 ← yolov8s 老师）

按官方文档 https://docs.ultralytics.com/zh/guides/knowledge-distillation/ 配置：学生起点 = n 系列最强（yolov8n-v6-1-best-pt），老师 = 已训好的 yolov8s-v1-1-pt，distill_model 指向老师。


In [ ]:
# 第1步：安装 YOLO 工具箱（ultralytics 库）
#   ! 开头表示在 Kaggle 终端直接执行系统命令
#   pip install = 用 pip 包管理器安装
#   ultralytics = YOLO 的官方 Python 库（训练/推理都靠它）
#   -q = quiet，静默安装，少打一堆无关日志
!pip install ultralytics -q


## 一、配置（每次训练只改这里）


In [ ]:
# 第2步：训练参数配置（想调训练就改这几行的数字）
EPOCHS   = 50      # 训练轮数：把整份数据来回学几遍（50 遍）
IMG_SIZE = 640     # 输入图片尺寸：每张图缩放到 640x640 再喂给模型
BATCH    = 16      # 每批几张图一起算（16 张；显存小就调小）
SEED     = 42      # 随机种子：固定它，每次训练随机过程可复现（结果可对照）
DIS      = 6.0     # 蒸馏系数：学生模仿老师答案的力度（官方默认 6.0，越大越向老师看齐）


## 二、自动定位学生/老师/数据包（按数据集 slug 区分；zip 会被 Kaggle 自动解压）


In [ ]:
# 第3步：导入工具模块（glob=按通配符找文件；os=操作系统路径操作）
import glob, os

# 定义一个小函数：从完整路径里抠出数据集名（slug）
#   因为上传的两个模型文件都叫 best.pt，文件名区分不了，只能看挂载目录名
def slug_of(p):
    # 把 Windows/linux 两种斜杠统一成 / 再按 / 拆成一段一段
    parts = p.replace("\\", "/").split("/")
    # 遍历每一段路径
    for i, x in enumerate(parts):
        # 找到 datasets 这一层，它后面第二个就是数据集 slug（如 yolov8s-v1-1-pt）
        if x == "datasets" and i + 2 < len(parts):
            # 返回 slug 作为这个模型的显示名
            return parts[i + 2]
    # 万一没找到 datasets 层，退回用文件名
    return os.path.basename(p)

# 找出 /kaggle/input 下所有 .pt 权重文件（学生、老师都在里面）
#   ** = 递归往下翻所有子目录
cands_pt = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
# 把找到的模型打印出来，方便确认挂载对不对
print("挂载的 pt:")
for p in cands_pt:
    print(" ", slug_of(p), "->", p)

# 学生 = 数据集 slug 里带 yolov8n 的那个（n系列最强，蒸馏要学它）
#   [0] = 取第一个（过滤后只剩一个）
STUDENT = [p for p in cands_pt if "yolov8n" in slug_of(p).lower()][0]
# 老师 = 数据集 slug 里带 yolov8s 的那个（已训好的 8s）
TEACHER = [p for p in cands_pt if "yolov8s" in slug_of(p).lower()][0]
# 打印确认选对了
print("学生:", STUDENT)
print("老师:", TEACHER)

# 找数据包：优先找 .zip；找不到说明 Kaggle 已自动解压
#   （Kaggle 上传 zip 会自动解压成目录，所以 zip 可能不存在）
cands_zip = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
if cands_zip:
    # 优先挑名字里带 colab 的 zip（就是训练数据包）
    colab_zips = [p for p in cands_zip if "colab" in p.lower()]
    # 有就取第一个（没有 colab 的取任意一个）
    ZIP = (colab_zips or cands_zip)[0]
    # 打印 zip 路径和大小（MB）
    print("数据包 zip:", ZIP, "%.1f MB" % (os.path.getsize(ZIP) / 1048576))
else:
    # 没有 zip：说明已经被自动解压了，后面直接用挂载目录
    print("无 zip（Kaggle 已自动解压），将直接使用挂载目录")


## 三、准备数据配置并蒸馏训练


In [ ]:
# 第4步：导入解压(zipfile)、读yaml(yaml)、YOLO 主类
import zipfile, yaml
from ultralytics import YOLO

# 解压目标目录（/kaggle/working 是可写区，解压到这里）
EXTRACT = "/kaggle/working/dataset"
# 先假设找不到数据目录
data_dir = None

if cands_zip:
    # 有 zip：先删掉上次解压的旧目录（避免残留干扰）
    if os.path.exists(EXTRACT):
        os.system(f"rm -rf {EXTRACT}")
    # 打开 zip 压缩包
    with zipfile.ZipFile(ZIP) as z:
        # 全部解压到 EXTRACT
        z.extractall(EXTRACT)
    # 在解压目录里翻找含 data.yaml 的文件夹（数据集的入口文件）
    for root, _, files in os.walk(EXTRACT):
        if "data.yaml" in files:
            # 找到了就记住这个目录并停下
            data_dir = root
            break
else:
    # 没 zip（已自动解压）：在 /kaggle/input 下找所有含 data.yaml 的目录
    data_dirs = [r for r, _, files in os.walk("/kaggle/input") if "data.yaml" in files]
    # 排除考试集目录（名字带 exam 的是考试包，训练要的是训练包）
    train_dirs = [d for d in data_dirs if "exam" not in d.lower()]
    # 取第一个训练数据目录
    data_dir = train_dirs[0]
    print("数据目录(已解压):", data_dir)

# 兜底检查：找不到就报错停下，别瞎跑
assert data_dir, "找不到 data.yaml"

# 读取数据包自带的 data.yaml（里面有 7 个类别名）
with open(os.path.join(data_dir, "data.yaml"), encoding="utf-8") as f:
    inner = yaml.safe_load(f)
# 取出类别名列表
names = inner.get("names")
# 兜底检查：没有类别名就报错
assert names, "data.yaml 缺少 names"

# 生成一份新的数据配置（Kaggle 上路径和本地不同，要重写 path）
DATA_YAML = "/kaggle/working/data_colab.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    # 写入：数据根目录 path、训练集路径、验证集路径、类别列表头
    f.write(f"path: {data_dir}\ntrain: images/train\nval: images/val\nnames:\n")
    # 如果 names 是字典格式（{0: 塑料瓶, ...}）就按 key: value 写
    if isinstance(names, dict):
        for k, v in names.items():
            f.write(f"  {k}: {v}\n")
    else:
        # 如果是列表格式（[塑料瓶, ...]）就按序号: 名字写
        for i, n in enumerate(names):
            f.write(f"  {i}: {n}\n")
# 打印配置路径确认
print("数据配置:", DATA_YAML)

# 训练结果文件夹的名字（结果会存到这里）
NAME = "tud_distill_v1.1"
# 创建 YOLO 模型对象，加载学生权重（n系列最强作为起点）
model = YOLO(STUDENT)
# 开始蒸馏训练（核心：distill_model=TEACHER 让老师当教练）
model.train(
    data=DATA_YAML,          # 用哪份数据训练
    epochs=EPOCHS,           # 训练轮数（上面配置的 50）
    imgsz=IMG_SIZE,          # 输入图片尺寸（640）
    batch=BATCH,             # 每批图片数（16）
    seed=SEED,               # 随机种子（42，可复现）
    dis=DIS,                 # 蒸馏强度（6.0，官方默认）
    device="0,1",            # 双卡：2 张 T4 并行（官方 cfg：device 多 GPU 逗号分隔，默认 None 只走单卡）
    distill_model=TEACHER,   # 【蒸馏关键】老师模型路径，训练时参考老师输出
    name=NAME,               # 结果目录名
    project="/kaggle/working/runs/detect",  # 结果放哪
)
# 打印训练完成和结果位置
print("训练完成，结果目录: /kaggle/working/runs/detect/" + NAME)


## 四、打包结果（只留核心文件，减小 output 同步体积）


In [ ]:
# 第5步：把训练结果打包成 zip 方便下载
import zipfile

# 结果目录（runs/detect 下叫 tud_distill_v1.1 的文件夹）
p   = f"/kaggle/working/runs/detect/{NAME}"
# 打包输出的文件名
out = f"/kaggle/working/{NAME}_results.zip"
# 创建一个 zip 压缩包（ZIP_DEFLATED = 标准压缩）
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    # 遍历结果目录下所有文件
    for root, _, files in os.walk(p):
        for fn in files:
            # 拼出完整路径
            fp = os.path.join(root, fn)
            # 跳过 val_batch 图片（一堆预测示例图，体积大没用，省下载时间）
            if "val_batch" in fn:
                continue
            # 把文件写进 zip，保留相对目录结构（方便解压后找）
            z.write(fp, os.path.relpath(fp, "/kaggle/working/runs/detect"))
# 打印结果包位置
print("结果包:", out)
# 提示去哪里下载
print("打包完成，去网页 Output 面板下载，或保存版本后命令行拉取。")
